# 02 — What we train, and how we judge it

Old CLM experiment: freeze the encoder and train projection heads on cached vectors.

New JevK5 experiment: freeze the merged backbone weights and train **new low-rank matrices inside its attention layers**. Gradients must pass through the network; cached CLM vectors cannot substitute for these forwards.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'skillret_decider').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Project:', ROOT.name)
print('No paid resources or real models are started by this notebook.')

In [ ]:
config = json.loads((ROOT / 'configs/pilot.json').read_text())
print('Starting checkpoint:', config['model'])
print('LoRA:', config['lora'])
print('Pilot limits:', config['train_limit'], 'train queries;', config['training']['max_steps'], 'updates')

## Soft-target cross-entropy

The model produces one logit per option in a single forward pass. This loss encourages probability on all labelled relevant options, rather than arbitrarily selecting the first positive.

In [ ]:
import torch
from skillret_decider.model import soft_target_ce
logits = torch.zeros((1, 3), requires_grad=True)
target = torch.tensor([[0.5, 0.5, 0.0]])
loss = soft_target_ce(logits, target)
loss.backward()
print('Initial loss:', float(loss.detach()))
print('Gradient:', logits.grad.tolist())
# Negative gradients for both positives encourage their logits to increase.

## What has actually been tested

The CPU toy is intentionally tiny. Its result is evidence for the training math and file round trip, **not a JevK5 accuracy or speed result**.

In [ ]:
toy = json.loads((ROOT / 'evidence/toy_lora_result.json').read_text())
print(json.dumps(toy, indent=2))

## RunPod pilot command — not executed here

```bash
python -m scripts.train --data workspace/data-pilot --config configs/pilot.json --out workspace/runs/pilot-001 --approve-gpu-run
```

Check real GPU loading, zero-LoRA parity, finite gradients, trainable names, memory, truncation, saved adapter reload and measured timing. The full run is a separate approval. A stopped training process does not stop Pod billing.

## The comparison that matters

| System | Purpose |
|---|---|
| BGE-M3 only | Strong retrieval baseline from the previous experiment |
| BGE-M3 + unadapted JevK5 | Does reranking already help? |
| BGE-M3 + SkillRet LoRA | Does adaptation add a genuine improvement? |

Use exactly the same natural candidates. Choose the checkpoint on validation Recall@5, with MRR@15 as tiebreaker; the unadapted model can remain the winner. Freeze the procedure before opening the official test.

Historical CLM `.3174` versus BGE `.5545` is useful context, not an automatically comparable new score.